Participant 1 Code

In [1]:
import yfinance as yf
import pandas as pd

ticker_symbol = "^FTSE"
start_date = "1999-01-01"
end_date = "2025-12-31"

ftse_data = yf.download(ticker_symbol, start=start_date, end=end_date, auto_adjust=True)

ftse_data.columns = ftse_data.columns.droplevel(1)

ftse_data.index = pd.to_datetime(ftse_data.index)

print(ftse_data.head())

ftse_data.to_csv("FTSE100 index.csv")

[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open      Volume
Date                                                                      
1999-01-04  5879.399902  5916.899902  5811.299805  5909.399902   689690000
1999-01-05  5958.200195  5980.500000  5875.799805  5882.299805  1033599000
1999-01-06  6148.799805  6157.399902  5968.899902  5968.899902   903848000
1999-01-07  6101.200195  6153.700195  6042.500000  6145.899902   650124000
1999-01-08  6147.200195  6195.600098  6114.799805  6115.399902   559612000


In [2]:

import pandas as pd
import numpy as np

df = pd.read_csv("FTSE100 index.csv", index_col=0)
df.index = pd.to_datetime(df.index)

df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
df['Rolling_Vol'] = df['Log_Return'].rolling(window=21).std() * np.sqrt(252)
df['Adaptive_Threshold'] = df['Rolling_Vol'].rolling(window=252).quantile(0.85)
df['Raw_Indicator'] = (df['Rolling_Vol'] > df['Adaptive_Threshold']).astype(int)
df['Crisis_Indicator'] = df['Raw_Indicator'].rolling(window=5, center=True).median().fillna(0).astype(int)
df_processed = df.dropna(subset=['Adaptive_Threshold'])

print(f"Total Trading Days: {len(df_processed)}")
print(f"Total Crisis Days: {df_processed['Crisis_Indicator'].sum()}")
print(f"Crisis Percentage: {(df_processed['Crisis_Indicator'].mean()*100):.2f}%")

df_processed.to_csv("FTSE100 Processed.csv")

Total Trading Days: 6546
Total Crisis Days: 1002
Crisis Percentage: 15.31%


In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.stats import (normaltest, skew, kurtosis,
                         shapiro, kstest, jarque_bera,
                         levene, mannwhitneyu, ttest_ind,
                         probplot, anderson)
import warnings
warnings.filterwarnings("ignore")

# ── Aesthetic config ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",  # White background
    "axes.facecolor":   "#ffffff",  # White plot area
    "axes.edgecolor":   "#d0d7de",  # Light gray border
    "axes.labelcolor":  "#24292f",  # Dark gray/black for labels (changed from white for legibility)
    "axes.titlecolor":  "#24292f",  # Dark gray/black for title
    "xtick.color":      "#57606a",  # Muted dark gray for ticks
    "ytick.color":      "#57606a",  # Muted dark gray for ticks
    "text.color":       "#24292f",  # Dark gray/black for general text
    "grid.color":       "#f6f8fa",  # Very light gray for grid lines
    "grid.linestyle":   "--",
    "grid.linewidth":   0.6,
    "legend.facecolor": "#ffffff",
    "legend.edgecolor": "#d0d7de",
    "font.family":      "monospace",
    "figure.dpi":       130,
})

CRISIS_COLOR   = "#ff7b72"   # red
NORMAL_COLOR   = "#3fb950"   # green
ACCENT_COLOR   = "#58a6ff"   # blue
WARN_COLOR     = "#d29922"   # amber
NEUTRAL_COLOR  = "#8b949e"   # grey
VOL_COLOR      = "#bc8cff"   # purple

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("FTSE100 Processed.csv", index_col=0)
df.index = pd.to_datetime(df.index, format='mixed', dayfirst=True)

# Keep only rows with the Adaptive_Threshold (same filter as preprocessing)
df_full = df.copy()
df = df.dropna(subset=["Adaptive_Threshold"]).copy()

crisis    = df[df["Crisis_Indicator"] == 1]
non_crisis = df[df["Crisis_Indicator"] == 0]

print("=" * 65)
print("  FTSE 100 — EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 65)

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 1 — Dataset Overview
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 1 — Dataset Overview")
print("-" * 50)
overview = {
    "Index":                    "FTSE 100 (^FTSE)",
    "Start Date":               str(df.index.min().date()),
    "End Date":                 str(df.index.max().date()),
    "Total Trading Days":       len(df),
    "Missing Values (Close)":   int(df["Close"].isna().sum()),
    "Crisis Days":              int(df["Crisis_Indicator"].sum()),
    "Non-Crisis Days":          int((df["Crisis_Indicator"] == 0).sum()),
    "Crisis %":                 f"{df['Crisis_Indicator'].mean()*100:.2f}%",
}
for k, v in overview.items():
    print(f"  {k:<32} {v}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 2 — Descriptive Statistics: Price
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 2 — Descriptive Statistics: Close Price")
print("-" * 50)
price_stats = df["Close"].describe()
extra = pd.Series({
    "skewness":  skew(df["Close"].dropna()),
    "kurtosis":  kurtosis(df["Close"].dropna()),
    "CV (%)":    df["Close"].std() / df["Close"].mean() * 100,
})
price_all = pd.concat([price_stats, extra])
for k, v in price_all.items():
    print(f"  {k:<20} {v:>12.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 3 — Descriptive Statistics: Log Returns
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 3 — Descriptive Statistics: Log Returns")
print("-" * 50)
ret = df["Log_Return"].dropna()
ret_stats = {
    "count":      len(ret),
    "mean":       ret.mean(),
    "std":        ret.std(),
    "min":        ret.min(),
    "25%":        ret.quantile(0.25),
    "50%":        ret.median(),
    "75%":        ret.quantile(0.75),
    "max":        ret.max(),
    "skewness":   skew(ret),
    "excess kurt":kurtosis(ret),
    "ann. mean":  ret.mean() * 252,
    "ann. vol":   ret.std() * np.sqrt(252),
}
for k, v in ret_stats.items():
    print(f"  {k:<20} {v:>14.6f}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 4 — Normality Tests on Log Returns
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 4 — Normality Tests: Log Returns")
print("-" * 50)
jb_stat,  jb_p  = jarque_bera(ret)
sw_stat,  sw_p  = shapiro(ret.sample(min(5000, len(ret)), random_state=42))
da_stat,  da_p  = normaltest(ret)
ks_stat,  ks_p  = kstest(ret, "norm", args=(ret.mean(), ret.std()))
print(f"  {'Test':<22} {'Statistic':>12} {'p-value':>12} {'Normal?':>10}")
print(f"  {'-'*58}")
for name, stat, p in [("Jarque-Bera", jb_stat, jb_p),
                       ("Shapiro-Wilk (5k)", sw_stat, sw_p),
                       ("D'Agostino-Pearson", da_stat, da_p),
                       ("Kolmogorov-Smirnov", ks_stat, ks_p)]:
    verdict = "NO" if p < 0.05 else "YES"
    print(f"  {name:<22} {stat:>12.4f} {p:>12.6f} {verdict:>10}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 5 — Rolling Volatility Descriptive Stats by Regime
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 5 — Rolling Volatility by Regime")
print("-" * 50)
for label, subset in [("Crisis", crisis), ("Non-Crisis", non_crisis), ("All", df)]:
    v = subset["Rolling_Vol"].dropna()
    print(f"\n  [{label}]")
    print(f"    n={len(v)}, mean={v.mean():.4f}, std={v.std():.4f}, "
          f"median={v.median():.4f}, min={v.min():.4f}, max={v.max():.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 6 — Statistical Tests: Crisis vs Non-Crisis Volatility
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 6 — Statistical Tests: Crisis vs Non-Crisis Volatility")
print("-" * 50)
c_vol  = crisis["Rolling_Vol"].dropna()
nc_vol = non_crisis["Rolling_Vol"].dropna()

t_stat, t_p     = ttest_ind(c_vol, nc_vol)
mw_stat, mw_p   = mannwhitneyu(c_vol, nc_vol, alternative="two-sided")
lev_stat, lev_p = levene(c_vol, nc_vol)

print(f"  {'Test':<28} {'Statistic':>12} {'p-value':>12} {'Significant?':>14}")
print(f"  {'-'*68}")
for name, stat, p in [("Welch's t-test", t_stat, t_p),
                       ("Mann-Whitney U", mw_stat, mw_p),
                       ("Levene (variance)", lev_stat, lev_p)]:
    sig = "YES ***" if p < 0.001 else ("YES *" if p < 0.05 else "NO")
    print(f"  {name:<28} {stat:>12.4f} {p:>12.6f} {sig:>14}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 7 — Annual Return Summary
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 7 — Annual Log-Return & Volatility Summary (selected years)")
print("-" * 65)
annual = df.groupby(df.index.year).agg(
    Ann_Return = ("Log_Return", lambda x: x.sum()),
    Ann_Vol    = ("Log_Return", lambda x: x.std() * np.sqrt(252)),
    Crisis_Pct = ("Crisis_Indicator", lambda x: x.mean() * 100),
    Trading_Days = ("Close", "count"),
).round(4)
print(annual.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 8 — Percentile Analysis: Log Returns
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 8 — Return Percentiles (Log Returns)")
print("-" * 50)
percentiles = [0.1, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9]
for p in percentiles:
    val = np.percentile(ret, p)
    print(f"  {p:>6.1f}th percentile: {val:>10.6f}")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 9 — Extreme Return Events (top 10 up & down)
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 9 — Top 10 Most Extreme Daily Returns")
print("-" * 50)
extreme_neg = df["Log_Return"].nsmallest(10)
extreme_pos = df["Log_Return"].nlargest(10)
print("\n  Largest DECLINES:")
for date, val in extreme_neg.items():
    regime = "CRISIS" if df.loc[date, "Crisis_Indicator"] == 1 else "normal"
    print(f"    {date.date()}  {val:>10.4f}  [{regime}]")
print("\n  Largest GAINS:")
for date, val in extreme_pos.items():
    regime = "CRISIS" if df.loc[date, "Crisis_Indicator"] == 1 else "normal"
    print(f"    {date.date()}  {val:>10.4f}  [{regime}]")

# ─────────────────────────────────────────────────────────────────────────────
# TABLE 10 — Autocorrelation of Returns and Abs Returns
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 TABLE 10 — Autocorrelation (lags 1-10)")
print("-" * 55)
print(f"  {'Lag':<6} {'Returns':>12} {'|Returns|':>12} {'Returns²':>12}")
print(f"  {'-'*44}")
for lag in range(1, 11):
    ac_r  = ret.autocorr(lag=lag)
    ac_ar = ret.abs().autocorr(lag=lag)
    ac_r2 = (ret**2).autocorr(lag=lag)
    print(f"  {lag:<6} {ac_r:>12.4f} {ac_ar:>12.4f} {ac_r2:>12.4f}")


# =====================================================================
#  CHARTS
# =====================================================================

# ── CHART 1 — Full Price History with Crisis Shading ─────────────────────────
print("\n🎨 Generating Chart 1: FTSE 100 Price History …")
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df.index, df["Close"], color=ACCENT_COLOR, lw=0.9, label="Close Price")
crisis_mask = df["Crisis_Indicator"] == 1
ax.fill_between(df.index, df["Close"].min(), df["Close"].max(),
                where=crisis_mask, color=CRISIS_COLOR, alpha=0.18, label="Crisis Period")
ax.set_title("FTSE 100 — Daily Close Price (1999–2025)  |  Crisis Periods Highlighted",
             fontsize=13, pad=12)
ax.set_xlabel("Date"); ax.set_ylabel("Index Level")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(loc="upper left")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("chart01_price_history.png", bbox_inches="tight")
plt.close()

# ── CHART 2 — Log Returns Time Series ────────────────────────────────────────
print("🎨 Generating Chart 2: Log Returns Time Series …")
fig, ax = plt.subplots(figsize=(16, 5))
colors = [CRISIS_COLOR if c == 1 else NEUTRAL_COLOR
          for c in df["Crisis_Indicator"]]
ax.bar(df.index, df["Log_Return"], color=colors, width=1.0, alpha=0.75)
ax.axhline(0, color="#f0f6fc", lw=0.6, ls="--")
ax.set_title("FTSE 100 — Daily Log Returns  |  Crisis Days in Red", fontsize=13, pad=12)
ax.set_xlabel("Date"); ax.set_ylabel("Log Return")
legend_elements = [Patch(facecolor=CRISIS_COLOR, label="Crisis"),
                   Patch(facecolor=NEUTRAL_COLOR, label="Non-Crisis")]
ax.legend(handles=legend_elements, loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("chart02_log_returns.png", bbox_inches="tight")
plt.close()

# ── CHART 3 — Rolling Volatility with Adaptive Threshold ─────────────────────
print("🎨 Generating Chart 3: Rolling Volatility & Threshold …")
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df["Rolling_Vol"],       color=VOL_COLOR,   lw=0.9, label="21-day Rolling Vol (Ann.)")
ax.plot(df.index, df["Adaptive_Threshold"],color=WARN_COLOR,  lw=1.1, ls="--", label="Adaptive Threshold (85th pct, 252-day)")
ax.fill_between(df.index, df["Rolling_Vol"], df["Adaptive_Threshold"],
                where=(df["Rolling_Vol"] > df["Adaptive_Threshold"]),
                color=CRISIS_COLOR, alpha=0.25, label="Threshold Breach")
ax.set_title("FTSE 100 — 21-Day Annualised Rolling Volatility vs Adaptive Threshold", fontsize=13, pad=12)
ax.set_xlabel("Date"); ax.set_ylabel("Annualised Volatility")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.legend(loc="upper left")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("chart03_rolling_vol_threshold.png", bbox_inches="tight")
plt.close()

# ── CHART 4 — Return Distribution: Histogram + KDE ───────────────────────────
print("🎨 Generating Chart 4: Return Distribution …")
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(ret, bins=150, density=True, color=ACCENT_COLOR, alpha=0.55, label="Observed")
xmin, xmax = ret.quantile(0.001), ret.quantile(0.999)
x = np.linspace(xmin, xmax, 500)
norm_pdf = stats.norm.pdf(x, ret.mean(), ret.std())
ax.plot(x, norm_pdf, color=WARN_COLOR, lw=2.0, ls="--", label="Normal PDF")
# KDE
kde = stats.gaussian_kde(ret)
ax.plot(x, kde(x), color=CRISIS_COLOR, lw=2.0, label="KDE")
ax.set_xlim(xmin, xmax)
ax.set_title("FTSE 100 — Distribution of Daily Log Returns vs Normal Distribution", fontsize=13, pad=12)
ax.set_xlabel("Log Return"); ax.set_ylabel("Density")
ax.legend()
ax.grid(True, alpha=0.4)
ax.text(0.01, 0.96,
        f"Skewness: {skew(ret):.4f}\nExcess Kurt: {kurtosis(ret):.4f}",
        transform=ax.transAxes,
        va="top",
        fontsize=9,
        color="Black", 
        bbox=dict(boxstyle="round,pad=0.4",
                  fc="#dfe1e6",
                  ec="#30363d"))
plt.tight_layout()
plt.savefig("chart04_return_distribution.png", bbox_inches="tight")
plt.close()

# ── CHART 5 — QQ Plot ────────────────────────────────────────────────────────
print("🎨 Generating Chart 5: QQ Plot …")
fig, ax = plt.subplots(figsize=(7, 6))
(osm, osr), (slope, intercept, r) = probplot(ret, dist="norm")
ax.scatter(osm, osr, s=4, color=ACCENT_COLOR, alpha=0.5, label="Observed Quantiles")
ax.plot(osm, slope * np.array(osm) + intercept, color=WARN_COLOR, lw=1.5, label="Normal Reference Line")
ax.set_title("Q-Q Plot — Log Returns vs Normal Distribution", fontsize=13, pad=12)
ax.set_xlabel("Theoretical Quantiles"); ax.set_ylabel("Sample Quantiles")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("chart05_qq_plot.png", bbox_inches="tight")
plt.close()

# ── CHART 6 — Crisis vs Non-Crisis Return Distributions ──────────────────────
print("🎨 Generating Chart 6: Crisis vs Non-Crisis Distributions …")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, subset, label, color in [
    (axes[0], crisis["Log_Return"].dropna(),     "Crisis",     CRISIS_COLOR),
    (axes[1], non_crisis["Log_Return"].dropna(), "Non-Crisis", NORMAL_COLOR),
]:
    ax.hist(subset, bins=80, density=True, color=color, alpha=0.6)
    x = np.linspace(subset.quantile(0.001), subset.quantile(0.999), 300)
    ax.plot(x, stats.norm.pdf(x, subset.mean(), subset.std()),
            color=WARN_COLOR, lw=2.0, ls="--", label="Normal PDF")
    ax.plot(x, stats.gaussian_kde(subset)(x), color=ACCENT_COLOR, lw=1.5, label="KDE")
    ax.set_title(f"{label} — Return Distribution", fontsize=12)
    ax.set_xlabel("Log Return"); ax.set_ylabel("Density")
    ax.text(0.03, 0.96,
            f"μ = {subset.mean():.5f}\nσ = {subset.std():.5f}\n"
            f"Skew = {skew(subset):.4f}\nKurt = {kurtosis(subset):.4f}",
            transform=ax.transAxes, va="top", fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.4", fc="#ebeff5", ec="#30363d"))
    ax.legend(fontsize=8); ax.grid(True, alpha=0.4)
plt.suptitle("Return Distribution — Crisis vs Non-Crisis Regimes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("chart06_regime_distributions.png", bbox_inches="tight")
plt.close()

# ── CHART 7 — Box Plots: Volatility by Regime ────────────────────────────────
print("🎨 Generating Chart 7: Box Plots …")
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, col, title, unit in [
    (axes[0], "Rolling_Vol",  "21-Day Rolling Volatility", "Annualised Vol"),
    (axes[1], "Log_Return",   "Daily Log Returns",         "Log Return"),
]:
    data_c  = crisis[col].dropna()
    data_nc = non_crisis[col].dropna()
    bp = ax.boxplot([data_nc, data_c],
                    labels=["Non-Crisis", "Crisis"],
                    patch_artist=True,
                    medianprops=dict(color="white", lw=2),
                    flierprops=dict(marker=".", ms=2, alpha=0.4),
                    whiskerprops=dict(lw=1.2),
                    capprops=dict(lw=1.2))
    bp["boxes"][0].set_facecolor(NORMAL_COLOR + "66")
    bp["boxes"][1].set_facecolor(CRISIS_COLOR + "66")
    for flier in bp["fliers"]:
        flier.set(markerfacecolor=WARN_COLOR, alpha=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(unit)
    ax.grid(True, axis="y", alpha=0.4)
plt.suptitle("Distribution Comparison — Crisis vs Non-Crisis", fontsize=13)
plt.tight_layout()
plt.savefig("chart07_boxplots.png", bbox_inches="tight")
plt.close()

# ── CHART 8 — Annual Volatility Heatmap ──────────────────────────────────────
print("🎨 Generating Chart 8: Annual Volatility Heatmap …")
df_monthly = df.copy()
df_monthly["Year"]  = df_monthly.index.year
df_monthly["Month"] = df_monthly.index.month

monthly_vol = df_monthly.groupby(["Year", "Month"])["Rolling_Vol"].mean().unstack()
monthly_vol.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                        "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(monthly_vol, cmap="YlOrRd", ax=ax, linewidths=0.3,
            linecolor="#21262d", annot=False,
            cbar_kws={"label": "Mean Rolling Volatility (Ann.)"})
ax.set_title("Monthly Average Annualised Volatility Heatmap (Year × Month)", fontsize=13, pad=12)
ax.set_xlabel("Month"); ax.set_ylabel("Year")
plt.tight_layout()
plt.savefig("chart08_vol_heatmap.png", bbox_inches="tight")
plt.close()

# ── CHART 9 — Annual Return Bar Chart with Crisis Overlay ────────────────────
print("🎨 Generating Chart 9: Annual Returns …")
fig, ax1 = plt.subplots(figsize=(16, 6))
bars_c  = annual["Ann_Return"] >= 0
bar_colors = [NORMAL_COLOR if x else CRISIS_COLOR for x in bars_c]
ax1.bar(annual.index, annual["Ann_Return"] * 100,
        color=bar_colors, alpha=0.8, width=0.6, label="Annual Log Return (%)")
ax1.axhline(0, color="#f0f6fc", lw=0.7, ls="--")
ax1.set_xlabel("Year"); ax1.set_ylabel("Annual Log Return (%)")
ax2 = ax1.twinx()
ax2.plot(annual.index, annual["Crisis_Pct"], color=WARN_COLOR,
         marker="o", ms=4, lw=1.5, label="Crisis Days %")
ax2.set_ylabel("Crisis Day %", color=WARN_COLOR)
ax2.tick_params(axis="y", colors=WARN_COLOR)
ax1.set_title("FTSE 100 — Annual Log Returns & Crisis Day Fraction", fontsize=13, pad=12)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
ax1.grid(True, axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig("chart09_annual_returns.png", bbox_inches="tight")
plt.close()

# ── CHART 10 — Autocorrelation Plots ─────────────────────────────────────────
print("🎨 Generating Chart 10: Autocorrelation …")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, series, label, color in [
    (axes[0], ret,       "Log Returns",          ACCENT_COLOR),
    (axes[1], ret.abs(), "Absolute Returns",     VOL_COLOR),
    (axes[2], ret**2,    "Squared Returns",      CRISIS_COLOR),
]:
    lags = range(1, 41)
    acf  = [series.autocorr(lag=l) for l in lags]
    ci   = 1.96 / np.sqrt(len(series))
    ax.bar(lags, acf, color=color, alpha=0.7, width=0.7)
    ax.axhline( ci, ls="--", color=WARN_COLOR, lw=1.0, label="95% CI")
    ax.axhline(-ci, ls="--", color=WARN_COLOR, lw=1.0)
    ax.axhline(0,   ls="-",  color="#f0f6fc",  lw=0.5)
    ax.set_title(f"ACF — {label}", fontsize=11)
    ax.set_xlabel("Lag (days)"); ax.set_ylabel("Autocorrelation")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle("Autocorrelation Functions — Returns, |Returns|, Returns²", fontsize=13)
plt.tight_layout()
plt.savefig("chart10_autocorrelation.png", bbox_inches="tight")
plt.close()

# ── CHART 11 — Rolling Statistics (Mean & Vol) ───────────────────────────────
print("🎨 Generating Chart 11: Rolling Statistics …")
roll252_mean = df["Log_Return"].rolling(252).mean() * 252
roll252_std  = df["Log_Return"].rolling(252).std() * np.sqrt(252)
roll63_std   = df["Log_Return"].rolling(63).std()  * np.sqrt(252)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
ax1.plot(df.index, roll252_mean * 100, color=ACCENT_COLOR, lw=0.9, label="252-day Rolling Annual Mean Return")
ax1.axhline(0, color="#f0f6fc", lw=0.5, ls="--")
ax1.fill_between(df.index, 0, roll252_mean * 100, alpha=0.15, color=ACCENT_COLOR)
ax1.set_ylabel("Ann. Mean Return (%)"); ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.4)

ax2.plot(df.index, roll252_std, color=VOL_COLOR,   lw=0.9, label="252-day Ann. Vol")
ax2.plot(df.index, roll63_std,  color=WARN_COLOR,  lw=0.9, alpha=0.7, label="63-day Ann. Vol")
crisis_mask = df["Crisis_Indicator"] == 1
ax2.fill_between(df.index, 0, roll252_std.max(),
                 where=crisis_mask, color=CRISIS_COLOR, alpha=0.15, label="Crisis")
ax2.set_ylabel("Annualised Volatility"); ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.4)
plt.suptitle("FTSE 100 — Rolling Annual Mean Return & Volatility (252-day & 63-day)", fontsize=13)
plt.tight_layout()
plt.savefig("chart11_rolling_stats.png", bbox_inches="tight")
plt.close()

# ── CHART 12 — Drawdown Analysis ─────────────────────────────────────────────
print("🎨 Generating Chart 12: Drawdown …")
rolling_max = df["Close"].cummax()
drawdown    = (df["Close"] - rolling_max) / rolling_max * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
ax1.plot(df.index, df["Close"], color=ACCENT_COLOR, lw=0.9)
ax1.fill_between(df.index, df["Close"].min(), df["Close"].max(),
                 where=(df["Crisis_Indicator"]==1), color=CRISIS_COLOR, alpha=0.15)
ax1.set_ylabel("Index Level")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax1.set_title("FTSE 100 — Price & Drawdown  |  Crisis Periods Shaded", fontsize=13, pad=12)
ax1.grid(True, alpha=0.4)

ax2.fill_between(df.index, drawdown, 0, color=CRISIS_COLOR, alpha=0.45)
ax2.plot(df.index, drawdown, color=CRISIS_COLOR, lw=0.7)
ax2.axhline(-20, ls="--", color=WARN_COLOR, lw=1.0, label="-20% Threshold")
ax2.axhline(-40, ls="--", color=CRISIS_COLOR, lw=1.0, label="-40% Threshold")
ax2.set_ylabel("Drawdown (%)"); ax2.set_xlabel("Date")
ax2.legend(); ax2.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("chart12_drawdown.png", bbox_inches="tight")
plt.close()

# ── CHART 13 — Volatility Distribution by Regime (Violin + Box) ──────────────
print("🎨 Generating Chart 13: Violin Plot …")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
data_plot = pd.DataFrame({
    "Volatility": pd.concat([c_vol, nc_vol]).values,
    "Regime":     (["Crisis"] * len(c_vol)) + (["Non-Crisis"] * len(nc_vol)),
})
palette = {"Crisis": CRISIS_COLOR, "Non-Crisis": NORMAL_COLOR}
sns.violinplot(data=data_plot, x="Regime", y="Volatility",
               palette=palette, ax=axes[0], inner="quartile", linewidth=1.2)
axes[0].set_title("Volatility Distribution — Violin Plot", fontsize=11)
axes[0].set_ylabel("21-Day Annualised Volatility")
axes[0].grid(True, axis="y", alpha=0.4)

ret_plot = pd.DataFrame({
    "Return": pd.concat([crisis["Log_Return"].dropna(),
                         non_crisis["Log_Return"].dropna()]).values,
    "Regime": (["Crisis"] * len(crisis["Log_Return"].dropna())) +
              (["Non-Crisis"] * len(non_crisis["Log_Return"].dropna())),
})
sns.violinplot(data=ret_plot, x="Regime", y="Return",
               palette=palette, ax=axes[1], inner="quartile", linewidth=1.2)
axes[1].set_title("Return Distribution — Violin Plot", fontsize=11)
axes[1].set_ylabel("Daily Log Return")
axes[1].grid(True, axis="y", alpha=0.4)
plt.suptitle("Distributional Shape — Crisis vs Non-Crisis (Violin)", fontsize=13)
plt.tight_layout()
plt.savefig("chart13_violin.png", bbox_inches="tight")
plt.close()

# ── CHART 14 — Scatter: Lagged Return vs Next-Day Return ─────────────────────
print("🎨 Generating Chart 14: Scatter Lag1 …")
df_sc = df[["Log_Return", "Crisis_Indicator"]].dropna().copy()
df_sc["Lag1"] = df_sc["Log_Return"].shift(1)
df_sc = df_sc.dropna()

fig, ax = plt.subplots(figsize=(8, 7))
colors_sc = [CRISIS_COLOR if c == 1 else NEUTRAL_COLOR + "88"
             for c in df_sc["Crisis_Indicator"]]
ax.scatter(df_sc["Lag1"], df_sc["Log_Return"],
           c=colors_sc, s=3, alpha=0.55)
m, b, r, p_val, se = stats.linregress(df_sc["Lag1"], df_sc["Log_Return"])
xf = np.linspace(df_sc["Lag1"].min(), df_sc["Lag1"].max(), 200)
ax.plot(xf, m * xf + b, color=WARN_COLOR, lw=1.8, label=f"OLS  r={r:.4f}")
ax.axhline(0, color="#f0f6fc", lw=0.5, ls="--")
ax.axvline(0, color="#f0f6fc", lw=0.5, ls="--")
ax.set_title("Lag-1 Return vs Next-Day Return  |  Crisis in Red", fontsize=12, pad=12)
ax.set_xlabel("Lag-1 Log Return"); ax.set_ylabel("Log Return (t)")
legend_elements = [Patch(facecolor=CRISIS_COLOR, label="Crisis"),
                   Patch(facecolor=NEUTRAL_COLOR, label="Non-Crisis"),
                   Line2D([0],[0], color=WARN_COLOR, lw=1.8, label=f"OLS r={r:.4f}")]
ax.legend(handles=legend_elements, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("chart14_lag_scatter.png", bbox_inches="tight")
plt.close()

# ── CHART 15 — Cumulative Log Return ─────────────────────────────────────────
print("🎨 Generating Chart 15: Cumulative Return …")
df["Cumulative_Return"] = df["Log_Return"].cumsum()
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df["Cumulative_Return"] * 100,
        color=ACCENT_COLOR, lw=0.9, label="Cumulative Log Return")
ax.fill_between(df.index, 0, df["Cumulative_Return"] * 100,
                where=(df["Crisis_Indicator"] == 1),
                color=CRISIS_COLOR, alpha=0.25, label="Crisis Period")
ax.axhline(0, color="#f0f6fc", lw=0.5, ls="--")
ax.set_title("FTSE 100 — Cumulative Log Return (1999–2025)  |  Crisis Shading", fontsize=13, pad=12)
ax.set_xlabel("Date"); ax.set_ylabel("Cumulative Log Return (%)")
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("chart15_cumulative_return.png", bbox_inches="tight")
plt.close()

# ── CHART BONUS 16 — Correlation Matrix ──────────────────────────────────────
print("🎨 Generating Chart 16: Correlation Matrix …")
cols_corr = ["Close", "Log_Return", "Rolling_Vol", "Adaptive_Threshold", "Crisis_Indicator"]
corr_matrix = df[cols_corr].dropna().corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, linecolor="#21262d",
            annot_kws={"size": 10},
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Correlation Matrix — Key EDA Variables", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("chart16_correlation_matrix.png", bbox_inches="tight")
plt.close()

  FTSE 100 — EXPLORATORY DATA ANALYSIS SUMMARY

📋 TABLE 1 — Dataset Overview
--------------------------------------------------
  Index                            FTSE 100 (^FTSE)
  Start Date                       2000-02-01
  End Date                         2025-12-30
  Total Trading Days               6546
  Missing Values (Close)           0
  Crisis Days                      1002
  Non-Crisis Days                  5544
  Crisis %                         15.31%

📋 TABLE 2 — Descriptive Statistics: Close Price
--------------------------------------------------
  count                   6546.0000
  mean                    6266.0529
  std                     1220.8785
  min                     3287.0000
  25%                     5430.0750
  50%                     6280.0000
  75%                     7185.2249
  max                     9940.7002
  skewness                   0.0507
  kurtosis                  -0.2223
  CV (%)                    19.4840

📋 TABLE 3 — Descriptive Statisti